# Mosaic integration: Mixed, with shared modality

Mosaic integration combines batches that share only some modalities, and every mosaic method reads ATAC as peaks. This tutorial is for an RNA + ADT batch, an RNA + ATAC batch and an RNA batch, so every batch holds RNA. It runs StabMap and UINMF on `D46mini`, 4,500 cells of the benchmark dataset `D46`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic_shared.ipynb).

## 1. Install

This cell installs `multibench-sc`.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.1" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D46mini` (4 MB) once. `mtb.env.install` downloads the environments for StabMap and UINMF. The download is 0.9 GB.

In [ ]:
import multibench as mtb
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column of the tables

mtb.data.fetch("D46mini")
METHODS = ["StabMap", "UINMF"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D46mini` and scores its output. The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D46mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D46mini", "mosaic", methods=METHODS, out_dir="out/D46mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size ranks the rows within each column and the colour scales each column, so a small gap can look large: `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw counts, with one `.h5ad` file per batch and modality and a cell type for each cell. Number the batches as below: RNA + ADT, then RNA + ATAC, then RNA. Give ATAC as peaks.

Every RNA file needs the same gene names.

Starting from something else? See the recipes for [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv) and [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:protein"` the matrix `obsm["protein"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

[The folder layout](https://dsichang.github.io/scMultiBench/tutorials/run/#folder-layout) lists the batch patterns and the methods that accept each one.

When the ADT file also holds a matrix in `obsm` under a name that does not start with `X_`, `export_dataset` asks which one is the protein matrix. `X_umap` and `X_pca` are left alone.

The folder stores each matrix dense, so drop the genes or peaks you do not need first.

</details>

So that the notebook runs as it is, the next cell writes five small example files, one per batch and modality, from the first 1,000 cells of each batch of `D46mini`.

In [ ]:
mtb.data.write_example("D46mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna1 = ad.read_h5ad("my_rna1.h5ad")     # <- batch 1: your RNA file
adt1 = ad.read_h5ad("my_adt1.h5ad")     # <- batch 1: proteins (ADT), same cells
rna2 = ad.read_h5ad("my_rna2.h5ad")     # <- batch 2: your RNA file
atac2 = ad.read_h5ad("my_atac2.h5ad")   # <- batch 2: peaks of the same cells
rna3 = ad.read_h5ad("my_rna3.h5ad")     # <- batch 3: your RNA file
print(rna1)
print(adt1)

Write the dataset folder. Change each name in quotes marked `# <-`. The printout above lists the names in two of your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYMOSAIC_SHARED"   # <- a new name for your own data
out = "mydata/" + NAME
celltype = "obs:celltype"   # <- the cell-type column of every file
# batch 1: RNA + ADT (one file: adt="obsm:protein")
mtb.io.export_dataset(rna1, out, rna="X", adt=adt1, labels=celltype,
                      batch_index=1, category="mosaic", overwrite=True)
# batch 2: RNA + ATAC peaks
mtb.io.export_dataset(rna2, out, rna="X", atac=atac2, atac_kind="peak",
                      labels=celltype,
                      batch_index=2, category="mosaic", overwrite=True)
# batch 3: RNA only
mtb.io.export_dataset(rna3, out, rna="X", labels=celltype,
                      batch_index=3, category="mosaic", overwrite=True)
mtb.scan("mydata/" + NAME, "mosaic", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "mosaic", methods=METHODS,
               out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

4 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic_shared_all.ipynb) runs them all. It downloads 3 envs, 2.5 GB on a computer without a GPU and 5.9 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.